In [4]:
# import the packages to create the data
import pandas as pd
import numpy as np
from faker import Faker
from datetime import timedelta
import random


In [5]:
# initialize the faker to make fake data
fake = Faker()
Faker.seed(42)
np.random.seed(42)


In [6]:
# This defines the categories that exist in the invented dataset for each of these variables
sex_at_birth = ["Male", "Female"]
gender = ["Male", "Female", "Transgender", "Non-binary"]
race_ethnicity = ["White", "Black", "Hispanic", "Asian", "Native American", "Other"]
transmission = ["MSM", "IDU", "Heterosexual", "MSM/IDU", "Perinatal", "Unknown"]
counties = [
    "Boone", "Cole", "Jackson", "Greene", "St. Louis City", "St. Louis County",
    "Clay", "Platte", "Cass", "Buchanan", "Jasper", "Cape Girardeau",
    "Christian", "Taney", "Phelps", "Callaway", "Audrain", "Howard",
    "Moniteau", "Osage", "Lafayette", "Johnson", "Pettis", "Saline",
    "Benton", "Miller", "Pulaski", "Texas", "Dent", "Laclede"
]
record_status = ["Confirmed", "Probable"]



In [7]:
# Here I weight the county variables for realistic simulation of data
# County weights (urban counties dominate)
county_weights = {
    "St. Louis City": 0.18,
    "St. Louis County": 0.22,
    "Jackson": 0.20,
    "Greene": 0.08,
    "Boone": 0.06,
    "Clay": 0.04,
    "Platte": 0.03,
    "Cass": 0.03,
    "Cole": 0.03,
    "Jasper": 0.03,
    "Other": 0.07
}

# Expand "Other" into remaining counties
other_counties = [
    "Buchanan", "Cape Girardeau", "Christian", "Taney", "Phelps",
    "Callaway", "Audrain", "Howard", "Moniteau", "Osage", "Lafayette",
    "Johnson", "Pettis", "Saline", "Benton", "Miller", "Pulaski",
    "Texas", "Dent", "Laclede"
]

county_choices = (
    ["St. Louis City", "St. Louis County", "Jackson", "Greene", "Boone",
     "Clay", "Platte", "Cass", "Cole", "Jasper"] +
    other_counties
)

county_probs = (
    [county_weights[c] for c in county_weights if c != "Other"] +
    [county_weights["Other"] / len(other_counties)] * len(other_counties)
)


In [8]:
# Here I weight the other variables
sex_weights = [0.78, 0.22]  # Male, Female

gender_probs = [0.75, 0.20, 0.03, 0.02]  # Male, Female, Trans, Non-binary

race_probs = [0.45, 0.32, 0.15, 0.05, 0.02, 0.01]  # White, Black, Hispanic, Asian, Native, Other

transmission_probs = [0.55, 0.15, 0.15, 0.05, 0.02, 0.08]  # MSM dominant


In [9]:
# this is so that the code later has gender being probabalistically related to sex
transmission_probs_by_sex = {
    "Male": {
        "MSM": 0.65,
        "Heterosexual": 0.15,
        "IDU": 0.12,
        "MSM/IDU": 0.05,
        "Perinatal": 0.01,
        "Unknown": 0.02
    },
    "Female": {
        "Heterosexual": 0.65,
        "IDU": 0.18,
        "MSM": 0.03,          # rare but possible
        "MSM/IDU": 0.01,
        "Perinatal": 0.10,
        "Unknown": 0.03
    }
}


In [10]:
def generate_case(case_id):
    dob = fake.date_of_birth(minimum_age=18, maximum_age=80)
    hiv_dx = fake.date_between(start_date='-20y', end_date='today')

    if random.random() < 0.6:
        aids_dx = None
    else:
        aids_dx = hiv_dx + timedelta(days=random.randint(-400, 3000))

    report_date = hiv_dx + timedelta(days=random.randint(-200, 1500))

    sex = random.choices(sex_at_birth, weights=sex_weights, k=1)[0]

    tx_dist = transmission_probs_by_sex[sex]
    transmission_category = random.choices(
        population=list(tx_dist.keys()),
        weights=list(tx_dist.values()),
        k=1
    )[0]

    if random.random() < 0.05:
        transmission_category = None

    return {
        "case_id": case_id,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "dob": dob,
        "sex_at_birth": sex,
        "current_gender": random.choices(gender, weights=gender_probs, k=1)[0],
        "race_ethnicity": random.choices(race_ethnicity + [None], weights=race_probs + [0.05], k=1)[0],
        "county": random.choices(county_choices, weights=county_probs, k=1)[0],
        "hiv_diagnosis_date": hiv_dx,
        "aids_diagnosis_date": aids_dx,
        "transmission_category": transmission_category,
        "last_cd4": random.choice([random.randint(20, 1500), None, -10]),
        "last_viral_load": random.choice([random.randint(20, 1_000_000), 0, -5]),
        "last_lab_date": hiv_dx + timedelta(days=random.randint(30, 3000)),
        "report_date": report_date,
        "record_status": random.choice(record_status)
    }


In [11]:
# a for loop that creates 5000 cases and assigns them to a dataframe
base_cases = [generate_case(i) for i in range(1, 5001)]
df = pd.DataFrame(base_cases)


In [12]:
# this creates duplicates at around 12%
dup_fraction = 0.12
dupes = df.sample(frac=dup_fraction).copy()

dupes["case_id"] += 10000

# Introduce common duplicate inconsistencies
dupes["last_name"] = dupes["last_name"].apply(
    lambda x: x[:-1] if len(x) > 3 else x
)

dupes["county"] = dupes["county"].sample(frac=1).values
dupes["record_status"] = "Duplicate"

df = pd.concat([df, dupes], ignore_index=True)


In [13]:
df

,case_id,first_name,last_name,dob,sex_at_birth,current_gender,race_ethnicity,county,hiv_diagnosis_date,aids_diagnosis_date,transmission_category,last_cd4,last_viral_load,last_lab_date,report_date,record_status
0,1,Angel,Hill,1988-07-05,Male,Male,Black,Jasper,2009-10-13,2011-09-17,MSM,373.0,197351,2012-08-29,2012-10-02,Probable
1,2,Jill,Rhodes,1960-03-08,Male,Male,White,Jackson,2010-09-29,2018-04-14,None,1321.0,0,2014-04-22,2014-05-05,Confirmed
2,3,Christopher,Henderson,2005-09-16,Male,Male,Black,St. Louis City,2024-07-20,None,MSM,136.0,-5,2031-12-12,2026-06-18,Probable
3,4,Jeffery,Wagner,1947-01-09,Male,Male,Black,St. Louis County,2009-03-08,None,IDU,-10.0,-5,2009-05-21,2012-12-18,Probable
4,5,James,Santos,1946-10-20,Male,Male,White,St. Louis County,2025-02-02,2028-01-28,MSM,NaN,584013,2028-01-12,2025-06-03,Probable
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5595,11026,Elizabeth,Davidso,1977-10-16,Male,Female,White,Jackson,2017-10-04,2025-01-29,MSM,-10.0,-5,2022-08-12,2017-07-21,Duplicate
5596,13110,Andrew,Welc,1964-04-05,Male,Male,White,St. Louis County,2016-04-02,2023-03-15,Heterosexual,-10.0,379684,2019-04-01,2019-02-16,Duplicate
5597,12111,Joseph,Richmon,1960-11-25,Male,Male,Black,Jackson,2006-08-18,None,MSM,NaN,513099,2014-08-05,2010-02-09,Duplicate
5598,11042,Michael,Schult,1945-02-22,Male,Male,Black,Jackson,2017-08-11,None,MSM,740.0,-5,2025-09-11,2019-12-05,Duplicate


In [14]:
# save the simulated data to a csv file
df = df.sample(frac=1).reset_index(drop=True)
df.to_csv("simulated_data.csv", index=False)
